## Welcome to Week 6 Day 3: Context Engineering with MCP

People used to talk about Prompt Engineering, but now it's given way to a new Skill "Context Engineering".

Philipp Schmid of Google DeepMind wrote the seminal post about Context Engineering:

https://www.philschmid.de/context-engineering

For this week, we will put Context Engineering into practice, heavily using MCP servers.

1. Long-term memory: a knowledge graph the agent writes to and reads back
2. Web search: fresh information from the live web
3. Agentic RAG: a vector store the agent fills from its own research, then searches
4. Integrations: connecting to a live external service, with a local fallback

In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio, create_static_tool_filter
import os
from pathlib import Path
from datetime import datetime
from IPython.display import Markdown, display

load_dotenv(override=True)

True

### A quick note for Windows

Launching a local MCP server from a notebook hits one rough edge on Windows. The server writes its startup output to stderr, but inside a Windows Jupyter kernel that stream has no real file handle behind it, so the launch fails with an `io.UnsupportedOperation: fileno` error, while Mac and Linux are unaffected.

The fix is to send the server's stderr to the null device, so it always has somewhere real to write. We do that once in the next cell, which lets every cell below use `MCPServerStdio` exactly as the OpenAI Agents SDK documents it. On Mac and Linux it costs nothing beyond keeping the server's startup banner out of the notebook.

In [2]:
# On Windows, a stdio MCP server started from a Jupyter kernel writes to a stderr stream with no
# real file descriptor and crashes with io.UnsupportedOperation: fileno. We send the server's
# stderr to the null device so it always has somewhere real to write, which lets every cell below
# use MCPServerStdio exactly as the OpenAI Agents SDK documents it. Mac and Linux are unaffected.
import functools
import subprocess
import agents.mcp.server

agents.mcp.server.stdio_client = functools.partial(agents.mcp.server.stdio_client, errlog=subprocess.DEVNULL)

## Part 2: Web search

A model only knows what it was trained on. Web search is the context source that keeps it current.

We use Tavily, a search API built for agents: it returns clean, ranked results an LLM can use directly. Tavily maintains its own MCP server, so connecting is a one-liner.

This needs a free API key:

1. Sign up at https://www.tavily.com
2. The free tier gives you 1,000 searches a month, with no credit card
3. Copy your API key (it starts with `tvly-`) and add it to your `.env` file:

`TAVILY_API_KEY=tvly-xxxx`

In [3]:
# Tavily MCP 서버를 stdio 방식으로 띄우기 위한 실행 파라미터 딕셔너리
tavily_params = {
    "command": "npx",                                   # 자식 프로세스로 실행할 명령어 (Node.js의 npx)
    "args": ["-y", "tavily-mcp@latest"],                # npx에 넘길 인자
                                                        # -y : 설치 여부 확인 프롬프트를 자동 승인
                                                        # tavily-mcp@latest : 실행할 npm 패키지와 버전 태그
    "env": {"TAVILY_API_KEY": os.getenv("TAVILY_API_KEY")},  # 자식 프로세스에 전달할 환경변수
                                                             # os.getenv로 현재 프로세스의 TAVILY_API_KEY 값을 읽어 전달
}

# MCPServerStdio: 로컬에서 MCP 서버를 자식 프로세스로 실행하고 stdin/stdout으로 JSON-RPC 통신
# client_session_timeout_seconds=60 : 서버 응답 대기 제한 시간(초)
# async with 블록을 벗어나면 서버 프로세스가 종료됨
async with MCPServerStdio(params=tavily_params, client_session_timeout_seconds=60) as server:
    tavily_tools = await server.list_tools()  # MCP의 tools/list 요청을 보내 서버가 제공하는 도구 목록을 받아옴

tavily_tools  # 받아온 도구 목록 확인 (블록을 벗어난 뒤라도 이미 받아온 데이터이므로 출력 가능)

[Tool(name='tavily_search', title=None, description='Search the web for current information on any topic. Use for news, facts, or data beyond your knowledge cutoff. Returns snippets and source URLs.', inputSchema={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'Search query'}, 'search_depth': {'type': 'string', 'enum': ['basic', 'advanced', 'fast', 'ultra-fast'], 'description': "The depth of the search. 'basic' for generic results, 'advanced' for more thorough search, 'fast' for optimized low latency with high relevance, 'ultra-fast' for prioritizing latency above all else", 'default': 'basic'}, 'topic': {'type': 'string', 'enum': ['general'], 'description': 'The category of the search. This will determine which of our agents will be used for the search', 'default': 'general'}, 'time_range': {'type': 'string', 'description': 'The time range back from the current date to include in the search results', 'enum': ['day', 'week', 'month', 'year']}, 'start_date':

Tavily's server offers several tools (search, extract, crawl, map, research). For this lab we only want plain web search, so we restrict the server to `tavily_search`. The OpenAI Agents SDK lets you hand an agent just the tools you choose with a static tool filter. Curating an agent's tools like this is itself context engineering.

In [4]:
# 에이전트의 역할을 정의하는 시스템 지침 (한글)
instructions = "당신은 웹을 검색해 정보를 찾고 핵심 내용을 간결하게 요약합니다."

# 에이전트에게 전달할 실제 요청문 (한글)
# f-string으로 현재 날짜를 문자열에 삽입 → 모델이 "최신"의 기준 시점을 알 수 있게 함
request = f"아마존(Amazon) 주가에 대한 최신 뉴스를 조사하고 향후 전망을 간략히 요약해 주세요. 참고로 오늘 날짜는 {datetime.now().strftime('%Y-%m-%d')}입니다."

# 사용할 LLM 모델 이름
model = "gpt-5-mini"

# MCP 서버가 제공하는 도구 중 일부만 노출시키는 정적(static) 필터 생성
# allowed_tool_names에 명시한 도구만 에이전트가 사용 가능
# tavily-mcp는 tavily_search 외에 tavily_extract, tavily_map, tavily_crawl 등도 제공하므로
# 검색 기능만 쓰도록 제한하는 목적
search_only = create_static_tool_filter(allowed_tool_names=["tavily_search"])

In [5]:
async with MCPServerStdio(params=tavily_params, client_session_timeout_seconds=60, tool_filter=search_only) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

요약 (2026-09-08 기준)

핵심 최신 뉴스
- 실적(2분기 2026): 매출 2,006억달러(+20% YoY), AWS 매출 422억달러(+37%), 순이익 626억달러(비영업성 이익 약 534억달러 — Anthropic 지분 재평가 영향). 회사 발표(About Amazon 등).
- 자본지출·AI 베팅: 2026년 약 2,000억달러 규모의 대규모 CAPEX 계획(주로 AI 데이터센터·커스텀 칩·위성 등). AWS의 AI 관련 연간화 수익(보고서상) 수십억 달러 규모로 가속.
- 위성사업: Globalstar 인수(약 115–116억달러 규모, Project Kuiper 보강) 보도 — 위성·스펙트럼 확보로 D2D·Kuiper 속도 확보.
- 규제·소송 리스크: 미국 FTC 등에서 광고 경매·요금 투명성 관련 조사·우려 보도(광고수익 주요 축이라 잠재적 제재·영향 가능).
- 운송·운영 이슈: 2026년 초·9월 초 보도된 Amazon 소유(계약) 화물기 사고(사상자 발생) 등 운영 리스크 관련 뉴스가 단기적으로 주가·이미지에 영향.
- 애널리스트·밸류에이션: 최근 종가 약 255–260달러대(초기 9월 데이터 기준), 컨센서스 목표주가 대략 320–330달러대(정보원에 따라 차이).

향후 전망(간단)
- 긍정 요인: AWS의 AI 수요 가속(고마진)과 커스텀 칩·클라우드·광범위 생태계(리테일·광고)가 장기 성장 동력. 위성·인프라 투자가 성공하면 신규 수익원·차별화 가능.
- 부정 요인: 대규모 CAPEX로 인한 단기·중기 현금흐름·자본효율 압박, 규제·광고 조사(벌금·비즈니스 제약), 경쟁(구글·MS·SpaceX 등)과 통합·실행 리스크. Anthropic 등 투자 관련 비현금성 이익에 따른 '실제 영업현금'과의 괴리 주의.
- 결론: 장기적(2–5년)으로는 AWS·AI 중심의 업사이드가 유효하나, 대규모 투자·규제 리스크로 단기 변동성 및 현금흐름 압박 가능성 높음. 투자 판단 시(1) CAPEX 집행 속도와 수익화(매출/백로그 대비) (2) 규제·소송 전개 (3) 분기별 FCF·영업이익 추이를 모니터링할 것을 권장.  

참고: 위 내용은 최신 보도와 회사 발표를 종합한 요약이며 투자 권유가 아닙니다. 더 자세한 기사·수치 원문을 원하시면 링크와 함께 제공해 드리겠습니다.

### Check out the trace

https://platform.openai.com/traces

## That's it for today

Four MCP servers, four kinds of context: long-term memory, web search, a knowledge base the agent builds itself, and a live integration. Choosing which sources an agent needs, and wiring them in, is much of what context engineering is in practice.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercises</h2>
            <span style="color:#ff7800;">Browse an MCP marketplace like glama.ai/mcp or smithery.ai and add another context source to this notebook, using one of today's four patterns.
            </span>
        </td>
    </tr>
</table>